In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys, platform, os
print(sys.executable)
print(platform.python_version())

/Users/AH45807/project/idiscovery-deep-research/.venv/bin/python
3.13.5


In [3]:
import pandas as pd

# Set the option to display all columns
pd.set_option('display.max_columns', None)

# You might also want to set max_colwidth to None to prevent column content truncation
pd.set_option('display.max_colwidth', None)
from datetime import datetime
import time
from collections import Counter
import pathlib
from pathlib import Path
from datetime import date
from collections import defaultdict
from dotenv import load_dotenv
# load_dotenv("/mnt/jupyter/hru/idiscovery-2pm-rule-extraction-hru/.env")

load_dotenv("/Users/AH45807/project/idiscovery-deep-research/.env")

True

In [4]:
from deep_research_utils.app_constant import AppConstants

2026-04-22 17:06:32,892 - policy_extractor.system - INFO - === Policy Extractor Logging Initialized ===
2026-04-22 17:06:32,893 - policy_extractor.system - INFO - Log directory: /Users/AH45807/project/idiscovery-deep-research/packages/core/src/deep_research_core/logs
2026-04-22 17:06:32,893 - policy_extractor.system - INFO - Max file size: 50.0MB
2026-04-22 17:06:32,894 - policy_extractor.system - INFO - Backup count: 10
2026-04-22 17:06:32,894 - policy_extractor.system - INFO - Console output enabled: True
2026-04-22 17:06:32,895 - policy_extractor.system - INFO - Console log level: INFO
2026-04-22 17:06:32,895 - policy_extractor.system - INFO - Console stream: stdout
2026-04-22 17:06:32,896 - policy_extractor.system - INFO - Process ID: 76542
2026-04-22 17:06:32,896 - policy_extractor.system - INFO - Component log levels:
2026-04-22 17:06:32,897 - policy_extractor.system - INFO -   policy_extractor.snowflake_store: WARNING
2026-04-22 17:06:32,897 - policy_extractor.system - INFO -   

In [5]:
from deep_research_utils import SnowparkHelper

snowpark_programmatic_connection_parameters = {
    "account" : os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_SECRET"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"]
}

snowpark = SnowparkHelper(
    connection_type="programmatic",
    batch_size=10000,        # Larger batches
    max_workers=6,           # Parallel processing
    enable_metrics=True,     # Performance tracking
    connection_pool_size=4,  # Connection pooling
    **snowpark_programmatic_connection_parameters
)

2026-04-22 17:06:36,325 - deep_research_utils.snowflake_helper - INFO - Snowflake session created successfully
2026-04-22 17:06:43,614 - deep_research_utils.snowflake_helper - INFO - Initialized connection pool with 3 additional sessions


In [6]:
from deep_research_utils import EHAPBase
EHAP = EHAPBase(base_url=os.environ.get("EHAP_BASE_URL"),
                client_id=os.environ.get("EHAP_CLIENT_ID"),
                client_secret=os.environ.get("EHAP_CLIENT_SECRET"),
                verify=os.environ.get("SSL_CERT_FILE"))

In [7]:
from langchain_openai import ChatOpenAI

In [8]:
# Chat model
model_medium_reasoning = ChatOpenAI(
    model="gpt-5.4",
    api_key=EHAP.get_token(),
    # output_version="completions/v1", # todo: check the version EHAP supports.
    # https://reference.langchain.com/python/langchain-openai/chat_models/base/ChatOpenAI
    extra_body={
        "reasoning_effort": "medium",  # Choices: "low", "medium", "high"
        "summary": None  # Choices: 'detailed', 'auto', or None 
    }
)

2026-04-22 17:06:44,323 - deep_research_utils.ehap - INFO - Requesting new access token from https://api.horizon.elevancehealth.com/v2/oauth2/token with client_id: BPWOQiwbyghnhWBaBzRYJhL3dU22bbb0
2026-04-22 17:06:44,501 - deep_research_utils.ehap - INFO - Access token generated successfully.
**TOKEN** **TOKEN** **TOKEN** 


/Users/AH45807/project/idiscovery-deep-research/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.horizon.elevancehealth.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [9]:
# ai_msg = model_medium_reasoning.invoke("What are the benefits of LangChain?")

# # Access the text content
# print(ai_msg.content)
# print(ai_msg.response_metadata)

# Step 1: Raja's Anomaly Detection

In [10]:
df_insights = pd.read_csv("/Users/AH45807/project/idiscovery-deep-research/packages/core/src/deep_research_core/coc_cmn_data_insght_stg-u01 2.csv")
print(df_insights.shape)
df_insights.head(2)


(24, 8)


,SNAP_YEAR_MNTH_NBR,TRND_TM_PRD_END_MNTH_NBR,TRND_TM_PRD_CD,LOB_CD,LOB_SHRT_DESC,STATSCL_MDL_CD,INSIGHTS_TYPE,JSON_TXT
0,202603,202512,R12,MCM006,Commercial,IP AUTH,KEY_INSIGHT,"{\n ""top_contributors"": {\n ""drgs"": [\n {\n ""insight"": ""Showed the highest growth in non-participating paid amounts."",\n ""name"": ""Vaginal Delivery without Sterilization or D&C with CC"",\n ""percentage_change"": ""+41%""\n },\n {\n ""insight"": ""Experienced significant growth in non-participating paid amounts."",\n ""name"": ""Vaginal Delivery without Sterilization or D&C without CC/MCC"",\n ""percentage_change"": ""+40%""\n },\n {\n ""insight"": ""Had a notable increase in non-participating paid amounts."",\n ""name"": ""Cesarean Section without Sterilization with MCC"",\n ""percentage_change"": ""+33%""\n }\n ],\n ""provider_trends"": [\n {\n ""insight"": ""Experienced a significant spike in non-participating paid amounts."",\n ""name"": ""JOHN PETER SMITH HOSPITAL"",\n ""percentage_change"": ""+4433%""\n },\n {\n ""insight"": ""Showed a substantial increase in non-participating paid amounts."",\n ""name"": ""EMORY DECATUR HOSPITAL"",\n ""percentage_change"": ""+356%""\n },\n {\n ""insight"": ""Had a notable rise in non-participating paid amounts."",\n ""name"": ""GRADY MEMORIAL HOSPITAL"",\n ""percentage_change"": ""+188%""\n }\n ],\n ""states"": [\n {\n ""insight"": ""Saw a significant increase in non-participating paid amounts."",\n ""name"": ""ME"",\n ""percentage_change"": ""+22%""\n },\n {\n ""insight"": ""Experienced a notable rise in non-participating paid amounts."",\n ""name"": ""CO"",\n ""percentage_change"": ""+13%""\n }\n ]\n },\n ""whats_happening"": ""Non-participating paid amounts are rising, primarily driven by a 3.35% increase in authorizations.""\n}"
1,202603,202512,YTD,MCM006,Commercial,IP AUTH,KEY_INSIGHT,"{\n ""top_contributors"": {\n ""drgs"": [\n {\n ""insight"": ""Had a substantial growth in non-participating paid amounts."",\n ""name"": ""Vaginal Delivery without Sterilization or D&C with CC"",\n ""percentage_change"": ""+41%""\n },\n {\n ""insight"": ""Showed significant growth in non-participating paid amounts."",\n ""name"": ""Vaginal Delivery without Sterilization or D&C without CC/MCC"",\n ""percentage_change"": ""+40%""\n },\n {\n ""insight"": ""Experienced notable growth in non-participating paid amounts."",\n ""name"": ""Cesarean Section without Sterilization with MCC"",\n ""percentage_change"": ""+33%""\n }\n ],\n ""provider_trends"": [\n {\n ""insight"": ""Experienced a significant spike in non-participating paid amounts."",\n ""name"": ""JOHN PETER SMITH HOSPITAL"",\n ""percentage_change"": ""+4433%""\n },\n {\n ""insight"": ""Showed a substantial increase in non-participating paid amounts."",\n ""name"": ""EMORY DECATUR HOSPITAL"",\n ""percentage_change"": ""+356%""\n },\n {\n ""insight"": ""Had a notable rise in non-participating paid amounts."",\n ""name"": ""GRADY MEMORIAL HOSPITAL"",\n ""percentage_change"": ""+188%""\n }\n ],\n ""states"": [\n {\n ""insight"": ""Saw a significant increase in non-participating paid amounts."",\n ""name"": ""ME"",\n ""percentage_change"": ""+22%""\n },\n {\n ""insight"": ""Experienced a notable rise in non-participating paid amounts."",\n ""name"": ""CO"",\n ""percentage_change"": ""+13%""\n }\n ]\n },\n ""whats_happening"": ""Non-participating paid amounts are rising, primarily driven by a 3.35% increase in authorizations.""\n}"


In [11]:
df_insights.INSIGHTS_TYPE.value_counts()

INSIGHTS_TYPE
KEY_INSIGHT    12
DEEP_DIVE      12
Name: count, dtype: int64

In [12]:
import json

In [13]:
# view sample insights
deep_dive = json.loads(df_insights.JSON_TXT.iloc[13])
print(deep_dive)


{
  "report_title": "IP AUTHORIZATION INSIGHTS - COMMERCIAL LOB, R12 PERIOD",
  "national_summary": {
    "description": "National Total Authorizations: 141,851"
  },
  "top_state_drivers": {
    "section_title": "TOP STATE DRIVERS",
    "states": [
      {
        "state_name": "CO",
        "overview": "CO has 5,898 auths out of 141,851 nationally (4.16% of the national total).",
        "medical_necessity_review_mix": "In CO, 41.88% of authorizations require Medical Necessity review (2,470 of 5,898), 14.99% do not require Medical Necessity review (884 of 5,898).",
        "service_driver": "In CO, of total 5,898 authorizations IP Med/Surg contributes 3,850 authorizations, accounting for 65.27%; IP BH contributes 1,469 authorizations, accounting for 24.90%; IP OB Dlvry NB contributes 524 authorizations, accounting for 8.89%; NF contributes 60 authorizations, accounting for 1.02%.",
        "authorization_status_mix": "In CO, Out of 5,898 authorizations received, 5,221 are in Approve

In [14]:
# Chat model
llm = ChatOpenAI(
    model=AppConstants.EHAP_LLM_MODEL,
    api_key=EHAP.get_token(),
    # output_version="completions/v1", # todo: check the version EHAP supports.
    # https://reference.langchain.com/python/langchain-openai/chat_models/base/ChatOpenAI
    extra_body={
        "reasoning_effort": "medium",  # Choices: "low", "medium", "high"
        "summary": None  # Choices: 'detailed', 'auto', or None 
    }
)

In [ ]:
# PATTERN_SYSTEM_PROMPT = """You are a conservative pattern-extraction engine.

# Your task is to analyze any INPUT_JSON dynamically and return a flat pattern table.
# Do NOT hardcode report sections, field names, or business-specific labels.
# Infer repeated structures, entities, metrics, and patterns directly from the input.

# Your output is for downstream processing, so reliability matters more than coverage.

# PRIMARY OBJECTIVE
# Return the smallest set of distinct, high-signal, non-overlapping patterns needed to summarize the input.

# A good pattern:
# - captures a meaningful repeated structure,
# - summarizes multiple facts into one insight, or
# - identifies a true outlier / exception.

# A bad pattern:
# - repeats the same idea for each driver type,
# - restates an obvious fact already covered by another row,
# - creates one row just because a section exists,
# - calls out low-signal zeros or missing values with little business value.

# OUTPUT RULES
# - Return ONLY valid JSON.
# - Do not return markdown.
# - The output must be a single JSON object with:
#   - "schema_summary"
#   - "pattern_table"
# - Each row in "pattern_table" must use the exact same keys.
# - If a field does not apply, use null.
# - If no defensible patterns exist, return an empty pattern_table.
# - Preserve exact evidence text where possible.

# STRICT NON-HALLUCINATION RULES
# - Do not invent facts, numbers, entities, categories, dimensions, causes, recommendations, or evidence labels.
# - Do not assume missing fields exist.
# - Do not create a pattern unless it is explicitly supported by the input.
# - Do not create “Policy Evidence”, “Claims Evidence”, or any other evidence category unless the source supports that framing.
# - When unsure about evidence category, use "General Evidence".
# - Do not convert weak hints into strong conclusions.
# - Do not add interpretations about why the pattern happened.
# - Do not add actions, recommendations, or root causes.

# ANTI-REPETITION / DEDUPLICATION RULES
# - Internally generate candidate patterns first, then deduplicate before producing final output.
# - Normalize each candidate into:
#   - dimension
#   - finding
#   - population covered
# - If two candidates express the same finding on the same dimension with only different supporting examples, merge them into one broader row.
# - Do NOT emit one row per driver family for the same normalized finding.
# - Prefer one cross-entity row over several near-duplicate rows.
# - If the same pattern appears across multiple entity types, merge it into one row with:
#   - entity_type = "mixed"
#   - scope = "cross_entity"
#   - covered entity types stored in attributes_json
# - Before emitting any row, test:
#   - "Does this row add materially new information not already captured by an existing row?"
#   - If NO, do not emit it.
# - If two rows would have very similar pattern_title, leadership_summary, and description, keep only the stronger one.
# - Do not let the number of rows track the number of entities, sections, or driver groups.

# NOVELTY / MATERIALITY RULES
# Only keep a separate row when at least one of these is true:
# - it covers multiple entities and compresses repeated evidence into one insight
# - it shows a materially different direction or dominant category
# - it shows meaningful contrast or spread
# - it captures an extreme concentration, extreme value, or outlier
# - it captures a unique exception to a broader pattern

# Treat these as materially distinct examples:
# - roughly 10+ percentage-point spread
# - 2x+ magnitude difference
# - 90%+ concentration
# - 100% concentration
# - unique zero / unique missing / unique exception with clear business relevance

# SUPPRESSION RULES
# Suppress or merge patterns that are:
# - obvious and universal in the sample without adding contrast
# - low-signal zeros repeated across many entities
# - simple restatements of the same dominant status or same dominant service across separate driver groups
# - narrower versions of a broader pattern already captured

# PATTERN BUDGET
# - Target 3 to 7 rows total.
# - Only exceed 7 rows when the input clearly contains more than 7 distinct, non-overlapping, material patterns.
# - Fewer rows is better than repetitive rows.

# ROW SCHEMA
# Each pattern row must have exactly these keys:

# 1. pattern_id
# A deterministic id if possible.

# 2. pattern_title
# Short title, noun-phrase style, suitable for business readers.

# 3. leadership_summary
# One concise leadership-friendly sentence.
# Requirements:
# - 12 to 30 words
# - direct and factual
# - include the most important quantified takeaway when possible

# 4. description
# One factual sentence with slightly more detail than leadership_summary.
# Do not speculate.

# 5. pattern_type
# A generic label such as:
# - volume_pattern
# - share_pattern
# - dominant_category
# - concentration
# - status_pattern
# - review_pattern
# - approval_pattern
# - denial_pattern
# - appeal_pattern
# - residual_pattern
# - missing_data_pattern
# - zero_value_pattern
# - cross_entity_pattern
# - outlier_pattern
# - other_pattern

# 6. entity_type
# Best inferred entity type from the source.
# Examples: state, provider, drg, product, market, segment.
# If the pattern spans multiple entity types, use "mixed".
# If unknown, use "unknown_entity_type".

# 7. entity_name
# The entity name if tied to one entity.
# If the pattern is cross-entity or report-level, use null.

# 8. scope
# One of:
# - entity
# - cross_entity
# - report

# 9. evidence
# An array of short evidence bullets sorted from biggest impact to lowest impact.
# Each evidence item must have:
# - rank: integer starting at 1
# - evidence_label: short free-text label inferred from the source, such as "Utilization Evidence", "Review Evidence", "Status Evidence", "General Evidence"
# - bullet: short factual bullet
# - quoted_numbers: array of exact number strings quoted from the source when available

# Evidence rules:
# - Keep bullets short.
# - Rank highest-impact evidence first.
# - Prefer bullets with the largest or most decision-relevant numbers first.
# - Quote numbers exactly as they appear when possible.
# - If there is only one evidence point, return a one-item array.
# - Merge similar evidence from duplicated candidates into the same row.
# - Do not fabricate evidence categories; use "General Evidence" when needed.

# 10. attributes_json
# A JSON object for extra structured details not covered above.
# This can include, when supported:
# - covered_entity_types
# - covered_entities
# - support_count
# - metric_name
# - metric_value
# - metric_unit
# - numerator
# - denominator
# - percentage
# - top_categories
# - status_mix
# - source_paths
# - comparison_basis
# - residual_value
# - missing_fields
# - related_entities
# - section_name
# - raw_snippets
# Use an empty object if nothing else is needed.

# SCHEMA SUMMARY
# Return a compact schema summary with:
# - detected_entity_types: array
# - common_keys_detected: array
# - repeated_shapes: array
# - notes: array

# PATTERN DETECTION RULES
# - Detect patterns from repeated structures, distributions, rankings, dominant categories, concentrations, counts, percentages, zero values, missing values, and cross-entity repetition.
# - Emit cross-entity rows for repeated findings that occur across multiple entities.
# - Emit entity-specific rows only for true outliers, exceptions, or uniquely important single-entity facts.
# - When a distribution has a clearly dominant category, emit a pattern only if it adds distinct information not already covered by another broader pattern.
# - When values are zero or missing, emit a row only if they are unique, material, or clearly notable.
# - Use attributes_json for extra detail rather than adding more top-level columns.

# RETURN FORMAT
# {
#   "schema_summary": {
#     "detected_entity_types": [],
#     "common_keys_detected": [],
#     "repeated_shapes": [],
#     "notes": []
#   },
#   "pattern_table": [
#     {
#       "pattern_id": "",
#       "pattern_title": "",
#       "leadership_summary": "",
#       "description": "",
#       "pattern_type": "",
#       "entity_type": "",
#       "entity_name": null,
#       "scope": "",
#       "evidence": [
#         {
#           "rank": 1,
#           "evidence_label": "General Evidence",
#           "bullet": "",
#           "quoted_numbers": []
#         }
#       ],
#       "attributes_json": {}
#     }
#   ]
# }
# """

In [ ]:
# PATTERN_USER_PROMPT_TEMPLATE = """Analyze the INPUT_JSON dynamically and return a tight generic pattern table.

# Requirements:
# - Do not rely on hardcoded field names, section names, or report templates.
# - Infer repeated entity structures and common patterns directly from the JSON.
# - Return the smallest set of distinct, defensible patterns needed to summarize the input.
# - Do not repeat the same pattern separately for each driver group or section.
# - Merge semantically similar findings into one broader row when possible.
# - Prefer cross-entity patterns over repeated section-level patterns.
# - Only emit entity-specific rows for real outliers, exceptions, or uniquely important facts.
# - Suppress low-signal or obvious universal patterns unless they add meaningful contrast.
# - Keep total rows low; do not let row count scale with the number of drivers.
# - For each pattern, include:
#   - a short business-friendly title
#   - a concise leadership-friendly summary
#   - a factual description
#   - a ranked evidence list from biggest impact to lowest impact
# - In the evidence list:
#   - use a short inferred evidence label
#   - keep bullets short
#   - quote numbers exactly where possible
#   - use "General Evidence" if no specific label is clearly supported
# - Put all extra details into attributes_json.
# - Do not hallucinate.
# - Do not speculate.
# - Do not invent evidence categories, causes, or recommendations.
# - Return only valid JSON.

# INPUT_JSON:
# {INPUT_JSON}
# """

In [17]:
# messages = [
#     {"role": "system", "content": PATTERN_SYSTEM_PROMPT},
#     {"role": "user", "content": PATTERN_USER_PROMPT_TEMPLATE.format(INPUT_JSON=deep_dive)},
# ]

# response = llm.invoke(messages)
# content = str(getattr(response, "content", response)).strip()

In [18]:
# # A deterministic post-filter after the LLM step would make this even safer: dedupe on a normalized key like (pattern_type, core_claim, dominant_dimension).
# df_patterns = pd.DataFrame(json.loads(content)["pattern_table"])
# print(df_patterns.shape)
# df_patterns.head(1)

In [ ]:
RISK_PATTERNS_SYSTEM_PROMPT = """You are a conservative healthcare risk-pattern extraction engine.

Your task is to analyze any INPUT_JSON dynamically and return a flat pattern table.
Do NOT hardcode report sections, field names, or business-specific labels.
Infer repeated structures, entities, metrics, and patterns directly from the input.

PRIMARY OBJECTIVE
Return the smallest set of distinct, high-signal, non-overlapping patterns that are plausibly relevant to payer risk.

This extractor is part of a healthcare risk detection pipeline.
Its purpose is NOT to summarize all patterns.
Its purpose is to surface only patterns that are plausibly relevant to:
- claims leakage
- rising medical cost
- utilization growth
- reimbursement risk
- provider contract risk
- benefit coverage risk
- state mandate / regulatory risk
- review-process risk
- denial / appeal exposure
- unexplained shifts that could materially affect insurer spend or liability

A good pattern:
- captures a meaningful repeated structure,
- summarizes multiple facts into one insight,
- identifies a true outlier / exception,
- and is plausibly relevant to payer cost, leakage, or exposure.

A bad pattern:
- repeats the same idea for each driver type,
- restates an obvious fact already covered by another row,
- creates one row just because a section exists,
- calls out low-signal zeros or descriptive facts with little payer-risk value.

OUTPUT RULES
- Return ONLY valid JSON.
- Do not return markdown.
- The output must be a single JSON object with:
  - "schema_summary"
  - "pattern_table"
- Each row in "pattern_table" must use the exact same keys.
- If a field does not apply, use null.
- If no defensible risk-relevant patterns exist, return an empty pattern_table.
- Preserve exact evidence text where possible.

STRICT NON-HALLUCINATION RULES
- Do not invent facts, numbers, entities, categories, dimensions, causes, recommendations, or evidence labels.
- Do not assume missing fields exist.
- Do not create a pattern unless it is explicitly supported by the input.
- Do not create “Policy Evidence”, “Claims Evidence”, or any other evidence category unless the source supports that framing.
- When unsure about evidence category, use "General Evidence".
- Do not convert weak hints into strong conclusions.
- Do not add interpretations about why the pattern happened.
- Do not add actions, recommendations, or root causes.
- Do not infer financial risk from a pattern unless the structure plausibly supports payer exposure.
- Do not assume a policy change, mandate change, provider contract issue, or benefit design issue occurred.

IMPORTANT TREND RULE
Do NOT claim a metric is "rising", "increasing", "growing", "worsening", or "declining" unless the input explicitly contains time-based evidence or a comparison period that supports that statement.

If the input does NOT contain time-series or benchmark comparison data:
- do NOT emit trend language
- instead use terms like:
  - elevated
  - concentrated
  - skewed
  - outlier
  - unusually high
  - heavily weighted
  - materially varied
  - potential risk signal

ANTI-REPETITION / DEDUPLICATION RULES
- Internally generate candidate patterns first, then deduplicate before producing final output.
- Normalize each candidate into:
  - dimension
  - finding
  - population covered
  - risk relevance
- If two candidates express the same finding on the same dimension with only different supporting examples, merge them into one broader row.
- Do NOT emit one row per driver family for the same normalized finding.
- Prefer one cross-entity row over several near-duplicate rows.
- If the same pattern appears across multiple entity types, merge it into one row with:
  - entity_type = "mixed"
  - scope = "cross_entity"
  - covered entity types stored in attributes_json
- Before emitting any row, test:
  - "Does this row add materially new information not already captured by an existing row?"
  - If NO, do not emit it.
- If two rows would have very similar pattern_title, leadership_summary, description, and risk_relevance, keep only the stronger one.
- Do not let the number of rows track the number of entities, sections, or driver groups.

RISK FILTERING RULES
Only emit a row if at least one of these is supported:
- unusually high or concentrated volume
- unusually high or concentrated share in a potentially cost-sensitive category
- high approval exposure in a sensitive service, provider, geography, or entity
- meaningful variation that could plausibly reflect reimbursement, policy, mandate, contract, or coverage differences
- denial, appeal, or overturn behavior that could affect spend or leakage
- high residual / missing / unclassified component that could hide exposure
- true outlier or exception with plausible payer risk relevance

Suppress patterns that are merely descriptive and do not clearly indicate risk relevance.

# RISK SIGNAL TYPES
# Use one of:
# - volume_increase
# - utilization_shift
# - unit_cost_increase
# - approval_exposure
# - service_mix_shift
# - high_cost_concentration
# - denial_overturn_exposure
# - appeal_exposure
# - provider_concentration_risk
# - geographic_concentration_risk
# - unexplained_outlier
# - missing_or_unclassified_risk
# - reimbursement_rule_sensitivity
# - policy_sensitive_pattern
# - contract_sensitive_pattern
# - mandate_sensitive_pattern
# - benefit_sensitive_pattern
# - other_risk_signal

PATTERN BUDGET
- Target 3 to 7 rows total.
- Only exceed 7 rows when the input clearly contains more than 7 distinct, material, non-overlapping risk patterns.
- Fewer rows is better than repetitive rows.

ROW SCHEMA
Each pattern row must have exactly these keys:

1. pattern_rank
A unique integer for each pattern, starting from 1. Ordered in most impactful pattern to be rank 1. Most impactful is where you would see large volume or variance in metrics.


2. pattern_title
Short title, noun-phrase style, suitable for business readers. Avoid vague term as listed values, use phrases such as "high cost concentration", "large <metric> variance", "large volume variance for <entity>".

3. leadership_summary
One concise leadership-friendly sentence.
Requirements:
- 12 to 30 words
- direct and factual
- include the most important quantified takeaway when possible

4. pattern_type
A generic label such as:
- volume_pattern
- share_pattern
- dominant_category
- concentration
- status_pattern
- review_pattern
- approval_pattern
- denial_pattern
- appeal_pattern
- residual_pattern
- missing_data_pattern
- zero_value_pattern
- cross_entity_pattern
- outlier_pattern
- other_pattern

4. risk_relevance
One concise sentence explaining why the pattern is worth downstream root-cause analysis, without guessing the cause.

8. entity_type
Best inferred entity type from the source.
Entity type is one of the following:
- SRVCAREA_ST_SHRT_DESC: SERVICE AREA STATE SHORT DESCRIPTION is the description of the State Code in the format of 2 character state abbreviation. eg: CA, VA, GA.
- FNL_DRG_NM: FINAL DRG NAME is the name for DIAGNOSIS RELATED GROUP (DRG) CODE. A DRG is a national coding scheme which classifies an inpatient stay based on diagnosis, procedure, discharge status, age and sex.
- 
It is a key value pair where key is "entity_type" and value is the entity type.
eg: {"state": ["CA", "VA"]}

9. entity_name
The entity name if tied to one entity.
If the pattern is cross-entity or report-level, use null.

10. scope
One of:
- entity
- cross_entity
- report

11. evidence
An array of short evidence bullets sorted from biggest impact to lowest impact.
Each evidence item must have:
- rank: integer starting at 1
- evidence_label: short free-text label inferred from the source, such as "Utilization Evidence", "Review Evidence", "Status Evidence", "General Evidence"
- bullet: short factual bullet
- quoted_numbers: array of exact number strings quoted from the source when available

Evidence rules:
- Keep bullets short.
- Rank highest-impact evidence first.
- Prefer bullets with the largest or most decision-relevant numbers first.
- Quote numbers exactly as they appear when possible.
- If there is only one evidence point, return a one-item array.
- Merge similar evidence from duplicated candidates into the same row.
- Do not fabricate evidence categories; use "General Evidence" when needed.

12. attributes_json
A JSON object for extra structured details not covered above.
This can include:
- covered_entity_types
- covered_entities
- support_count
- metric_name
- metric_value
- metric_unit
- numerator
- denominator
- percentage
- top_categories
- status_mix
- source_paths
- comparison_basis
- residual_value
- missing_fields
- related_entities
- section_name
- raw_snippets
Use an empty object if nothing else is needed.

SCHEMA SUMMARY
Return a compact schema summary with:
- detected_entity_types: array
- common_keys_detected: array
- repeated_shapes: array
- notes: array

RETURN FORMAT
{
  "schema_summary": {
    "detected_entity_types": [],
    "common_keys_detected": [],
    "repeated_shapes": [],
    "notes": []
  },
  "pattern_table": [
    {
      "pattern_id": "",
      "pattern_title": "",
      "leadership_summary": "",
      "description": "",
      "pattern_type": "",
      "risk_signal_type": "",
      "risk_relevance": "",
      "entity_type": "",
      "entity_name": null,
      "scope": "",
      "evidence": [
        {
          "rank": 1,
          "evidence_label": "General Evidence",
          "bullet": "",
          "quoted_numbers": []
        }
      ],
      "attributes_json": {}
    }
  ]
}
"""

In [20]:
RISK_PATTERNS_USER_PROMPT_TEMPLATE = """Analyze the INPUT_JSON dynamically and return a tight generic pattern table for healthcare payer risk detection.

Requirements:
- Do not rely on hardcoded field names, section names, or report templates.
- Infer repeated entity structures and common patterns directly from the JSON.
- Return the smallest set of distinct, defensible, risk-relevant patterns needed to summarize the input.
- Only include patterns that could plausibly matter for claims leakage, rising cost, utilization growth, reimbursement exposure, provider contract exposure, benefit coverage exposure, mandate/regulatory exposure, or review-process risk.
- Do not repeat the same pattern separately for each driver group or section.
- Merge semantically similar findings into one broader row when possible.
- Prefer cross-entity patterns over repeated section-level patterns.
- Only emit entity-specific rows for real outliers, exceptions, or uniquely important facts.
- Suppress low-signal or obvious universal patterns unless they add meaningful payer-risk relevance.
- Keep total rows low; do not let row count scale with the number of drivers.
- For each pattern, include:
  - a short business-friendly title
  - a concise leadership-friendly summary
  - a factual description
  - a risk_signal_type
  - a concise risk_relevance sentence
  - a ranked evidence list from biggest impact to lowest impact
- In the evidence list:
  - use a short inferred evidence label
  - keep bullets short
  - quote numbers exactly where possible
  - use "General Evidence" if no specific label is clearly supported
- Put all extra details into attributes_json.
- Do not hallucinate.
- Do not speculate.
- Do not invent causes, trends, evidence categories, or recommendations.
- Do not say a metric is rising or increasing unless the input explicitly contains time-based evidence.
- Return only valid JSON.

INPUT_JSON:
{INPUT_JSON}
"""

In [21]:
messages = [
    {"role": "system", "content": RISK_PATTERNS_SYSTEM_PROMPT},
    {"role": "user", "content": RISK_PATTERNS_USER_PROMPT_TEMPLATE.format(INPUT_JSON=deep_dive)},
]

response = llm.invoke(messages)
content = str(getattr(response, "content", response)).strip()

In [22]:
# A deterministic post-filter after the LLM step would make this even safer: dedupe on a normalized key like (pattern_type, core_claim, dominant_dimension).
df_risk_patterns = pd.DataFrame(json.loads(content)["pattern_table"])
print(df_risk_patterns.shape)
df_risk_patterns.head(1)

(4, 12)


,pattern_id,pattern_title,leadership_summary,description,pattern_type,risk_signal_type,risk_relevance,entity_type,entity_name,scope,evidence,attributes_json
0,pat_drg_ob_delivery_approval_concentration,OB delivery DRG approval concentration,"Three obstetric DRGs account for 7,068 listed authorizations, all 100.00% IP OB Dlvry NB, with 62.99%–68.34% not requiring Medical Necessity review.","The listed obstetric DRGs are entirely concentrated in IP OB Dlvry NB and show approved shares of 79.79%–87.96%, No Decision shares of 4.52%–7.47%, and Partial shares of 5.35%–10.11%.",concentration,approval_exposure,A large block of authorization volume is concentrated in one service family with substantial approval exposure and limited medical-necessity gating.,drg,NaN,cross_entity,"[{'rank': 1, 'evidence_label': 'General Evidence', 'bullet': 'Vaginal Delivery without Sterilization or D&C without CC/MCC has 4,093 auths, is 100.00% IP OB Dlvry NB, and reports 68.34% not requiring Medical Necessity review.', 'quoted_numbers': ['4,093', '100.00%', '68.34%']}, {'rank': 2, 'evidence_label': 'General Evidence', 'bullet': 'Cesarean Section without Sterilization without CC/MCC has 1,570 auths, is 100.00% IP OB Dlvry NB, and reports 67.26% not requiring Medical Necessity review with 87.96% Approved.', 'quoted_numbers': ['1,570', '100.00%', '67.26%', '87.96%']}, {'rank': 3, 'evidence_label': 'General Evidence', 'bullet': 'Vaginal Delivery without Sterilization or D&C with CC has 1,405 auths, is 100.00% IP OB Dlvry NB, with 79.79% Approved, 10.11% Partial, and 7.47% No Decision.', 'quoted_numbers': ['1,405', '100.00%', '79.79%', '10.11%', '7.47%']}]","{'covered_entity_types': ['drg'], 'covered_entities': ['Vaginal Delivery without Sterilization or D&C without CC/MCC', 'Cesarean Section without Sterilization without CC/MCC', 'Vaginal Delivery without Sterilization or D&C with CC'], 'support_count': 3, 'metric_name': 'listed_authorizations', 'metric_value': 7068, 'metric_unit': 'authorizations', 'top_categories': ['IP OB Dlvry NB'], 'source_paths': ['top_drg_drivers.drgs']}"


In [23]:
df_risk_patterns.leadership_summary

0                    Three obstetric DRGs account for 7,068 listed authorizations, all 100.00% IP OB Dlvry NB, with 62.99%–68.34% not requiring Medical Necessity review.
1    Listed providers range from 17.64% to 82.89% requiring Medical Necessity review, and some have only 34.79%–38.96% classified across the two reported review buckets.
2                                GRADY MEMORIAL HOSPITAL shows 18.41% Partial status on 516 authorizations, roughly double the other listed providers’ 4.93%–9.43% range.
3                                                 CO has 24 appeals from 138 denials (17.39%) and 3 overturns, the largest denial-related volume among the listed states.
Name: leadership_summary, dtype: str

In [24]:
s = ""
for _, row in df_risk_patterns.iterrows():
    s += f"{row.leadership_summary}\n"
    for e in row.evidence:
        s += f"- {e['bullet']}\n"
    s += "-" * 80 + "\n"
print(s)

Three obstetric DRGs account for 7,068 listed authorizations, all 100.00% IP OB Dlvry NB, with 62.99%–68.34% not requiring Medical Necessity review.
- Vaginal Delivery without Sterilization or D&C without CC/MCC has 4,093 auths, is 100.00% IP OB Dlvry NB, and reports 68.34% not requiring Medical Necessity review.
- Cesarean Section without Sterilization without CC/MCC has 1,570 auths, is 100.00% IP OB Dlvry NB, and reports 67.26% not requiring Medical Necessity review with 87.96% Approved.
- Vaginal Delivery without Sterilization or D&C with CC has 1,405 auths, is 100.00% IP OB Dlvry NB, with 79.79% Approved, 10.11% Partial, and 7.47% No Decision.
--------------------------------------------------------------------------------
Listed providers range from 17.64% to 82.89% requiring Medical Necessity review, and some have only 34.79%–38.96% classified across the two reported review buckets.
- GRADY MEMORIAL HOSPITAL reports 17.64% requiring Medical Necessity review and 21.32% not requiri

In [25]:
df_insights.INSIGHTS_TYPE.value_counts()

INSIGHTS_TYPE
KEY_INSIGHT    12
DEEP_DIVE      12
Name: count, dtype: int64

In [26]:
import traceback

In [27]:
op = []
for idx, insight in df_insights[df_insights.INSIGHTS_TYPE == 'DEEP_DIVE'].copy()[0:2].iterrows():
    deep_dive = insight.JSON_TXT
    try:
        messages = [
            {"role": "system", "content": RISK_PATTERNS_SYSTEM_PROMPT},
            {"role": "user", "content": RISK_PATTERNS_USER_PROMPT_TEMPLATE.format(INPUT_JSON=deep_dive)},
        ]
        response = llm.invoke(messages)
        content = str(getattr(response, "content", response)).strip()
        df_risk_patterns = pd.DataFrame(json.loads(content)["pattern_table"])
        s = ""
        for _, row in df_risk_patterns.iterrows():
            s += f"{row.leadership_summary}\n"
            for e in row.evidence:
                s += f"- {e['bullet']}\n"
            s += "-" * 80 + "\n"
        insight['PATTERN'] = s
    except:
        print(f"Error processing insight row: {idx}")
        print(traceback.format_exc())
        insight['PATTERN'] = ""
    op.append(insight)
    
df_output = pd.DataFrame(op)
df_output.shape

(2, 9)

In [29]:
df_output.to_excel("2026-06-23_patterns_output.xlsx", index=False)

In [32]:
df_risk_patterns[["leadership_summary", "evidence"]]

,leadership_summary,evidence
0,"GRADY lists only '91' require and '110' do not require review out of '516', and a top DRG classifies only '1,759' plus '323' of '3,397' approvals.","[{'rank': 1, 'evidence_label': 'Review Evidence', 'bullet': 'CO lists '2,470' requiring review and '884' not requiring review out of '5,898' total authorizations.', 'quoted_numbers': ['2,470', '884', '5,898', '41.88%', '14.99%']}, {'rank': 2, 'evidence_label': 'Approval Evidence', 'bullet': 'Vaginal Delivery without Sterilization or D&C without CC/MCC lists '1,759' medical-necessity approvals and '323' administrative approvals out of '3,397' approved authorizations.', 'quoted_numbers': ['1,759', '323', '3,397', '51.78%', '9.51%']}, {'rank': 3, 'evidence_label': 'Review Evidence', 'bullet': 'GRADY MEMORIAL HOSPITAL lists '91' requiring review and '110' not requiring review out of '516' total authorizations.', 'quoted_numbers': ['91', '110', '516', '17.64%', '21.32%']}, {'rank': 4, 'evidence_label': 'Review Evidence', 'bullet': 'EMORY DECATUR HOSPITAL lists '74' requiring review and '53' not requiring review out of '365' total authorizations.', 'quoted_numbers': ['74', '53', '365', '20.27%', '14.52%']}]"
1,"All three listed DRGs are '100.00%' IP OB Dlvry NB, with '79.79%' to '87.96%' approved and '62.99%' to '68.34%' not requiring review.","[{'rank': 1, 'evidence_label': 'Service Evidence', 'bullet': 'Vaginal Delivery without Sterilization or D&C without CC/MCC: '4,093' auths, '100.00%' IP OB Dlvry NB, '68.34%' do not require review, '83.00%' approved.', 'quoted_numbers': ['4,093', '100.00%', '68.34%', '83.00%']}, {'rank': 2, 'evidence_label': 'Service Evidence', 'bullet': 'Cesarean Section without Sterilization without CC/MCC: '1,570' auths, '100.00%' IP OB Dlvry NB, '67.26%' do not require review, '87.96%' approved.', 'quoted_numbers': ['1,570', '100.00%', '67.26%', '87.96%']}, {'rank': 3, 'evidence_label': 'Service Evidence', 'bullet': 'Vaginal Delivery without Sterilization or D&C with CC: '1,405' auths, '100.00%' IP OB Dlvry NB, '62.99%' do not require review, '79.79%' approved.', 'quoted_numbers': ['1,405', '100.00%', '62.99%', '79.79%']}, {'rank': 4, 'evidence_label': 'Status Evidence', 'bullet': 'The same DRGs also show decision friction, including '7.26%' No Decision and '7.16%' Partial, '4.52%' No Decision and '5.35%' Partial, and '7.47%' No Decision and '10.11%' Partial.', 'quoted_numbers': ['7.26%', '7.16%', '4.52%', '5.35%', '7.47%', '10.11%']}]"
2,"GRADY MEMORIAL HOSPITAL has '95' partial authorizations, or '18.41%' of '516', the highest partial share among listed providers.","[{'rank': 1, 'evidence_label': 'Status Evidence', 'bullet': 'GRADY MEMORIAL HOSPITAL has '95' Partial authorizations, '18.41%' of '516' total.', 'quoted_numbers': ['95', '18.41%', '516']}, {'rank': 2, 'evidence_label': 'Approval Evidence', 'bullet': 'GRADY MEMORIAL HOSPITAL has '411' Approved authorizations, '79.65%' of total.', 'quoted_numbers': ['411', '79.65%']}, {'rank': 3, 'evidence_label': 'General Evidence', 'bullet': 'Other listed provider partial shares are '9.43%' at CEDARS-SINAI MEDICAL CENTER, '8.46%' at JOHN PETER SMITH HOSPITAL, and '4.93%' at EMORY DECATUR HOSPITAL.', 'quoted_numbers': ['9.43%', '8.46%', '4.93%']}]"
3,"CO has the largest denial and appeal volume in the report: '138' not approved, '24' appealed, and '3' overturned out of '5,898' auths.","[{'rank': 1, 'evidence_label': 'Denial Evidence', 'bullet': 'CO has '138' denied authorizations; '24' were appealed, '3' overturned, and '15' upheld.', 'quoted_numbers': ['138', '24', '3', '15', '17.39%', '2.17%', '10.87%']}, {'rank': 2, 'evidence_label': 'Status Evidence', 'bullet': 'CO has '5,221' Approved authorizations ('88.52%') and '485' Partial authorizations ('8.22%') out of '5,898' total.', 'quoted_numbers': ['5,221', '88.52%', '485', '8.22%', '5,898']}, {'rank': 3, 'evidence_label': 'Denial Evidence', 'bullet': 'ME shows smaller state denial activity with '24' denied, '3' appealed, and '0.00